# Can we predict late deliveries?
Protocol: `protocol.yaml`. Temporal holdout (`holdout.csv`), 5-split walk-forward CV on training rows, selection by mean CV ROC AUC, holdout scored once.

Feature decision: `claim_paid_eur` is compensation paid on a claim *settled for this consignment*, i.e. recorded after the outcome, so it is not known at prediction time and is excluded (leakage). `consignment` (identifier) and `dispatched_on` (split only) are excluded per protocol.

In [ ]:
import json, hashlib, numpy as np, pandas as pd, yaml
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
proto = yaml.safe_load(open('protocol.yaml'))
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest() == proto['dataset']['sha256']
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest() == proto['split']['holdout_sha256']
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==proto['dataset']['rows'] and len(hold_idx)==proto['split']['holdout_rows']
is_hold = np.zeros(len(df),bool); is_hold[hold_idx]=True
df['dispatched_on']=pd.to_datetime(df['dispatched_on'])
train = df[~is_hold].sort_values('dispatched_on', kind='stable')
hold = df[is_hold]
print(len(train), len(hold), train.dispatched_on.max(), hold.dispatched_on.min())
print(train.late_delivery.mean(), hold.late_delivery.mean())

In [ ]:
y_col='late_delivery'
FEATURES=['carrier','distance_km','weight_kg','parcels']   # claim_paid_eur excluded: recorded after the outcome
num=['distance_km','weight_kg','parcels']; cat=['carrier']
def pre(scale):
    return ColumnTransformer([('num', StandardScaler() if scale else 'passthrough', num),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
# all candidates named before any scoring
candidates = {
 'baseline_prior': Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1': Pipeline([('p',pre(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1': Pipeline([('p',pre(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'rf_depth6': Pipeline([('p',pre(False)),('m',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,random_state=0))]),
 'hgb_small': Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

In [ ]:
X=train[FEATURES]; y=train[y_col].to_numpy()
tscv=TimeSeriesSplit(n_splits=proto['cv']['n_splits'])
cv_folds=[]; cv_scores={k:[] for k in candidates}
train_rows=train.index.to_numpy()   # 0-based positions in data.csv (default RangeIndex)
for tr,va in tscv.split(X):
    cv_folds.append([train_rows[tr].tolist(),train_rows[va].tolist()])
    for k,m in candidates.items():
        mm=clone(m).fit(X.iloc[tr],y[tr])   # preprocessing fitted inside the fold
        cv_scores[k].append(float(roc_auc_score(y[va],mm.predict_proba(X.iloc[va])[:,1])))
summary=pd.DataFrame(cv_scores).T; summary['mean']=summary.mean(axis=1)
print(summary.round(4))
chosen=summary['mean'].idxmax(); print('chosen:',chosen)

In [ ]:
final=clone(candidates[chosen]).fit(X,y)
p=final.predict_proba(hold[FEATURES])[:,1]
holdout_auc=float(roc_auc_score(hold[y_col],p))   # scored once
print('holdout roc_auc', holdout_auc)
evidence={'features':FEATURES,
 'candidates':{k:str(v) for k,v in candidates.items()},
 'cv_folds':cv_folds,'cv_scores':cv_scores,'chosen':chosen,
 'holdout_rows':hold_idx.tolist(),'holdout_scores':{chosen:holdout_auc}}
json.dump(evidence,open('evidence.json','w'))